# ClinSense — BERT Fine-tuning (Partial)

**Run this notebook in Google Colab** for GPU access.

Fine-tunes bert-base-uncased for **Medical Specialty Classification** (8 classes, MTSamples).
Freezes all layers except classifier + last 2 encoder layers. No LoRA.

Metrics: Micro F1, Macro F1, Precision, Recall.

## 1. Setup

In [ ]:
!pip install -q transformers torch datasets accelerate scikit-learn pandas matplotlib

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    matthews_corrcoef, cohen_kappa_score, classification_report, confusion_matrix
)
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
import torch

## 2. Load Data (Filtered 8-Class Dataset)

**Upload:** `data/processed/mtsamples_classification_filtered.csv` to `/content/` as `mtsamples_classification_filtered.csv`

- 8 classes: Cardiovascular/Pulmonary, Gastroenterology, Neurology, Obstetrics/Gynecology, Orthopedic, Radiology, SOAP/Chart/Progress Notes, Urology
- 1911 samples

In [ ]:
import os

# Prefer filtered 8-class dataset (optimal filtering from scripts/optimal_filter_and_train.py)
FILTERED_PATH = "/content/mtsamples_classification_filtered.csv"
FALLBACK_PATH = "/content/mtsamples_classification.csv"

df = None
if os.path.exists(FILTERED_PATH):
    df = pd.read_csv(FILTERED_PATH, encoding="utf-8", on_bad_lines="skip")
    print("Loaded filtered 8-class dataset (1911 samples)")
elif os.path.exists(FALLBACK_PATH):
    df = pd.read_csv(FALLBACK_PATH, encoding="utf-8", on_bad_lines="skip")
    print("Loaded fallback preprocessed data")
else:
    raise FileNotFoundError(
        "Upload mtsamples_classification_filtered.csv to /content/\n"
        "Run: python scripts/optimal_filter_and_train.py locally, then upload the filtered CSV."
    )

for c in ["Unnamed: 0", "0"]:
    if c in df.columns:
        df = df.rename(columns={c: "note_id"})
text_col = "transcription" if "transcription" in df.columns else "text"
df["transcription"] = df[text_col].fillna("").astype(str).str.strip()
df["medical_specialty"] = df["medical_specialty"].fillna("").astype(str).str.strip()
df = df[df["transcription"].str.len() >= 50].copy()
df = df[df["medical_specialty"].str.len() > 0].copy()

df["text"] = df["transcription"]
label2id = {k: i for i, k in enumerate(sorted(df["medical_specialty"].unique()))}
id2label = {v: k for k, v in label2id.items()}
num_labels = len(label2id)
print(f"Classes: {num_labels}, Samples: {len(df)}")
print("Classes:", list(label2id.keys()))
df["label"] = df["medical_specialty"].map(label2id)

In [ ]:
from sklearn.model_selection import train_test_split

# 70/15/15 stratified split (matches optimal_filter_and_train.py)
X = df["text"]
y = df["label"]
X_train, X_rest, y_train, y_rest = train_test_split(X, y, test_size=0.30, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_rest, y_rest, test_size=0.5, stratify=y_rest, random_state=42)

# Ensure labels are integers (0 to num_labels-1)
y_train = np.array(y_train, dtype=np.int64)
y_val = np.array(y_val, dtype=np.int64)
y_test = np.array(y_test, dtype=np.int64)
y_train_int = y_train.tolist()
y_val_int = y_val.tolist()
y_test_int = y_test.tolist()

print(f"Train: {len(X_train)} (70%), Val: {len(X_val)} (15%), Test: {len(X_test)} (15%)")
print(f"Label range: {y_train.min()}-{y_train.max()}, unique: {np.unique(y_train)}")

In [ ]:
# Data ready. X_train, y_train (oversampled), X_val, y_val, X_test, y_test

## 3. BERT for Classification (Partial Fine-tune)

In [ ]:
# Proven setup (69.3% in 3 epochs) — NO LoRA
MODEL_NAME = "bert-base-uncased"
MAX_LEN = 512
BATCH = 8
GRAD_ACCUM = 2
LR = 3e-5
EPOCHS = 15
WARMUP_STEPS = 50
WEIGHT_DECAY = 0.01

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)

# Freeze all EXCEPT classifier + last 2 encoder layers
for name, param in model.named_parameters():
    if "encoder.layer.10" in name or "encoder.layer.11" in name or "classifier" in name or "pooler" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
pct = 100 * trainable / total
print(f"Trainable: {trainable:,} / {total:,} ({pct:.1f}%)")

In [ ]:
print(f"Num labels: {model.config.num_labels}")

In [ ]:
# Standard Trainer (no WeightedTrainer, no label smoothing)

In [ ]:
# Trainer expects "labels" (plural)
def tokenize_fn(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LEN,
        padding="max_length",
        return_tensors=None,
    )

train_ds = Dataset.from_dict({"text": X_train.tolist(), "labels": y_train_int})
val_ds = Dataset.from_dict({"text": X_val.tolist(), "labels": y_val_int})
test_ds = Dataset.from_dict({"text": X_test.tolist(), "labels": y_test_int})

train_ds = train_ds.map(tokenize_fn, batched=True, remove_columns=["text"])
val_ds = val_ds.map(tokenize_fn, batched=True, remove_columns=["text"])
test_ds = test_ds.map(tokenize_fn, batched=True, remove_columns=["text"])

# Remove token_type_ids (critical fix for BioBERT)
if "token_type_ids" in train_ds.column_names:
    train_ds = train_ds.remove_columns(["token_type_ids"])
    val_ds = val_ds.remove_columns(["token_type_ids"])
    test_ds = test_ds.remove_columns(["token_type_ids"])
    print("Removed token_type_ids")

In [ ]:
# Data ready

In [ ]:
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.argmax(preds, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "micro_f1": f1_score(labels, preds, average="micro", zero_division=0),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        "weighted_f1": f1_score(labels, preds, average="weighted", zero_division=0),
        "micro_precision": precision_score(labels, preds, average="micro", zero_division=0),
        "macro_precision": precision_score(labels, preds, average="macro", zero_division=0),
        "micro_recall": recall_score(labels, preds, average="micro", zero_division=0),
        "macro_recall": recall_score(labels, preds, average="macro", zero_division=0),
        "hamming_loss": 1 - accuracy_score(labels, preds),
        "matthews_corrcoef": matthews_corrcoef(labels, preds),
        "cohen_kappa": cohen_kappa_score(labels, preds),
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="./clinsense_bert",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LR,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    lr_scheduler_type="cosine",
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    load_best_model_at_end=True,
    metric_for_best_model="eval_micro_f1",
    greater_is_better=True,
    fp16=True,
    logging_steps=20,
    report_to="none",
)

from transformers import Trainer, EarlyStoppingCallback

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
)

trainer.train()

In [ ]:
# Evaluate on test set

In [ ]:
# 1. Trainable params vs total
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
params_pct = 100 * trainable / total
print("=" * 60)
print("TRAINABLE PARAMS")
print("=" * 60)
print(f"  Trainable: {trainable:,} / {total:,} ({params_pct:.2f}%)")

# 2. Evaluate on test set
eval_results = trainer.evaluate(test_ds)
preds = trainer.predict(test_ds)
y_pred = np.argmax(preds.predictions, axis=1)
y_true = preds.label_ids
label_names = [id2label[i] for i in range(num_labels)]

# 3. Micro F1, Macro F1, Weighted F1
micro_f1 = eval_results.get("eval_micro_f1", f1_score(y_true, y_pred, average="micro", zero_division=0))
macro_f1 = eval_results.get("eval_macro_f1", f1_score(y_true, y_pred, average="macro", zero_division=0))
weighted_f1 = eval_results.get("eval_weighted_f1", f1_score(y_true, y_pred, average="weighted", zero_division=0))

print("\n" + "=" * 60)
print("TEST METRICS")
print("=" * 60)
print(f"  Micro F1:    {micro_f1:.4f}")
print(f"  Macro F1:    {macro_f1:.4f}")
print(f"  Weighted F1: {weighted_f1:.4f}")

# 4. Full classification report
print("\n" + "=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(y_true, y_pred, target_names=label_names, zero_division=0))

# 5. Save confusion matrix
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
cm = confusion_matrix(y_true, y_pred, labels=list(range(num_labels)))
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(num_labels))
ax.set_yticks(range(num_labels))
ax.set_xticklabels(label_names, rotation=45, ha="right")
ax.set_yticklabels(label_names)
for i in range(num_labels):
    for j in range(num_labels):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=9)
ax.set_title("Confusion Matrix: BERT (partial)")
plt.tight_layout()
plt.savefig("/content/confusion_matrix_bert.png", dpi=150, bbox_inches="tight")
plt.close()
print("Confusion matrix saved to /content/confusion_matrix_bert.png")

# 6. Final comparison table
print("\n" + "=" * 60)
print("FINAL COMPARISON")
print("=" * 60)
print("| Model              | Micro F1 | Macro F1 | Params Trained |")
print("|--------------------|----------|----------|----------------|")
print("| TF-IDF + LR        | 67.9%    | 68.3%    | N/A            |")
print("| TF-IDF + SVM       | 65.9%    | 66.3%    | N/A            |")
print("| bert-base (no LoRA)| 69.3%    | 67.0%    | 13.5%          |")
print(f"| BERT (partial)     | {micro_f1*100:.1f}%    | {macro_f1*100:.1f}%    | {params_pct:.1f}%           |")

In [ ]:
# Save model for deployment
model.save_pretrained("./clinsense_bert_final")
tokenizer.save_pretrained("./clinsense_bert_final")

# Download: from google.colab import files; files.download("./clinsense_bert_final")

## 4. (Optional) NER

For NER, use a NER dataset (e.g. BC5CDR) or fine-tune on token-level labels.
MTSamples does not have NER annotations. Example using HuggingFace BC5CDR:

```python
# pip install seqeval
# from datasets import load_dataset
# ds = load_dataset("tner/bc5cdr")
# ... token classification with LoRA
```

For now, use **scispaCy** (see `src/ner/scispacy_ner.py`) for entity extraction from MTSamples.